In [0]:
SILVER_BASE_PATH = "/Volumes/misha_azure/default/global_sales/silver"
GOLD_BASE_PATH   = "/Volumes/misha_azure/default/global_sales/gold"

In [0]:
try:
    sales_silver = spark.read.format("delta") \
        .load(f"{SILVER_BASE_PATH}/sales")

    products_silver = spark.read.format("delta") \
        .load(f"{SILVER_BASE_PATH}/products")

    stores_silver = spark.read.format("delta") \
        .load(f"{SILVER_BASE_PATH}/stores")

    returns_silver = spark.read.format("delta") \
        .load(f"{SILVER_BASE_PATH}/returns")

    customers_silver = spark.read.format("delta") \
        .load(f"{SILVER_BASE_PATH}/customers")

    targets_silver = spark.read.format("delta") \
        .load(f"{SILVER_BASE_PATH}/sales_targets")
except Exception as e:
    print("Error while loading silver tables: ",str(e))
    raise

Gold Table-1 (Sales Performance)

In [0]:
sales_selected = sales_silver.select(
    "order_date",
    "region",
    "order_id",
    "quantity",
    "revenue"
)

In [0]:
from pyspark.sql.functions import year, month

sales_time = sales_selected \
    .withColumn("order_year", year("order_date")) \
    .withColumn("order_month", month("order_date"))

In [0]:
#gold_sales_performance table
from pyspark.sql.functions import sum as sum_, countDistinct, avg

gold_sales_performance = sales_time.groupBy(
    "order_year",
    "order_month",
    "region"
).agg(
    countDistinct("order_id").alias("total_orders"),
    sum_("quantity").alias("total_quantity_sold"),
    sum_("revenue").alias("total_revenue"),
    avg("revenue").alias("avg_order_value")
)

In [0]:
gold_sales_performance.write.format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_BASE_PATH}/gold_sales_performance")
print("gold_sales_performance table written")

gold_sales_performance table written


In [0]:
gold_sales_performance.show(5)

+----------+-----------+------+------------+-------------------+-------------+------------------+
|order_year|order_month|region|total_orders|total_quantity_sold|total_revenue|   avg_order_value|
+----------+-----------+------+------------+-------------------+-------------+------------------+
|      2024|          7| SOUTH|          55|                175|    2339110.0| 42529.27272727273|
|      2025|         12| NORTH|         132|                387|    5170037.0| 39166.94696969697|
|      2023|         12|  EAST|          12|                 32|     380468.0|31705.666666666668|
|      2024|         11| NORTH|         147|                453|    5828525.0| 39649.82993197279|
|      2024|          4|  WEST|          91|                271|    3729236.0| 40980.61538461538|
+----------+-----------+------+------------+-------------------+-------------+------------------+
only showing top 5 rows


In [0]:
gold_sales_performance.count()

100

Gold Table-2(Product Performance)

In [0]:
sales_products = sales_silver.join(
    products_silver,
    on="product_id",
    how="inner"
)

In [0]:
sales_products_sel = sales_products.select(
    "product_id",
    "product_name",
    "category",
    "sub_category",
    "quantity",
    "revenue",
    "cost_price"
)

In [0]:
from pyspark.sql.functions import col

sales_products_enriched = sales_products_sel \
    .withColumn(
        "total_cost",
        col("quantity") * col("cost_price")
    ) \
    .withColumn(
        "profit",
        col("revenue") - (col("quantity") * col("cost_price"))
    )

In [0]:
#gold-product-performance
gold_product_performance = sales_products_enriched.groupBy(
    "product_id",
    "product_name",
    "category",
    "sub_category"
).agg(
    sum_("quantity").alias("total_quantity_sold"),
    sum_("revenue").alias("total_revenue"),
    sum_("total_cost").alias("total_cost"),
    sum_("profit").alias("total_profit")
)

In [0]:
from pyspark.sql.functions import when

gold_product_performance = gold_product_performance.withColumn(
    "profit_margin_pct",
    when(col("total_revenue") > 0,
         (col("total_profit") / col("total_revenue")) * 100
    ).otherwise(0)
)

In [0]:
gold_product_performance.write.format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_BASE_PATH}/gold_product_performance")
print("gold_product_performance table written")

gold_product_performance table written


In [0]:
gold_product_performance.show(5)

+----------+---------------+--------------+-------------+-------------------+-------------+----------+------------+------------------+
|product_id|   product_name|      category| sub_category|total_quantity_sold|total_revenue|total_cost|total_profit| profit_margin_pct|
+----------+---------------+--------------+-------------+-------------------+-------------+----------+------------+------------------+
|     P1162|Nihil Household|       GROCERY|    HOUSEHOLD|                 85|     754562.0|  340595.0|    413967.0| 54.86189338980759|
|     P1064|Eius Appliances|HOME & KITCHEN|   APPLIANCES|                 92|    1231194.0|  779792.0|    451402.0| 36.66375892020267|
|     P1100|   Natus Mobile|   ELECTRONICS|MOBILE PHONES|                 78|     761560.0|  708630.0|     52930.0| 6.950207468879668|
|     P1204|Veritatis Decor|HOME & KITCHEN|        DECOR|                120|     439537.0|  367080.0|     72457.0|16.484846554442516|
|     P1238|    Harum Decor|HOME & KITCHEN|        DECO

Gold Table-3 (Regional Performance)

In [0]:
sales_stores = sales_silver.alias("s").join(
    stores_silver.select(
        col("store_id"),
        col("store_name"),
        col("region").alias("store_region")   
    ).alias("st"),
    col("s.store_id") == col("st.store_id"),
    "inner"
).select(
    col("s.order_id"),
    col("s.revenue"),
    col("st.store_id"),
    col("st.store_name"),
    col("st.store_region").alias("region")   
)

In [0]:
sales_region_agg = sales_stores.groupBy(
    "region",
    "store_id",
    "store_name"
).agg(
    countDistinct("order_id").alias("total_orders"),
    sum_("revenue").alias("total_revenue")
)


In [0]:
returns_with_region = returns_silver.alias("r").join(
    sales_silver.select("order_id", "store_id").alias("s"),
    "order_id",
    "inner"
).join(
    stores_silver.select(
        col("store_id"),
        col("region").alias("store_region")
    ).alias("st"),
    "store_id",
    "inner"
).select(
    col("st.store_region").alias("region"),
    col("store_id"),
    col("refund_amount"),
    col("order_id")
)

In [0]:
returns_region_agg = returns_with_region.groupBy(
    "region",
    "store_id"
).agg(
    sum_("refund_amount").alias("total_refund_amount"),
    countDistinct("order_id").alias("returned_orders")
)


In [0]:
regional_performance = sales_region_agg.join(
    returns_region_agg,
    on=["region", "store_id"],
    how="left"
).fillna({
    "total_refund_amount": 0,
    "returned_orders": 0
})

In [0]:
gold_region_performance = regional_performance \
    .withColumn(
        "return_rate_pct",
        when(col("total_orders") > 0,
             (col("returned_orders") / col("total_orders")) * 100
        ).otherwise(0)
    ) \
    .withColumn(
        "net_revenue",
        col("total_revenue") - col("total_refund_amount")
    )

In [0]:
gold_region_performance = regional_performance \
    .withColumn(
        "return_rate_pct",
        when(col("total_orders") > 0,
             (col("returned_orders") / col("total_orders")) * 100
        ).otherwise(0)
    ) \
    .withColumn(
        "net_revenue",
        col("total_revenue") - col("total_refund_amount")
    )


In [0]:
gold_region_performance.write.format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_BASE_PATH}/gold_region_performance")
print("gold_region_performance table written")

gold_region_performance table written


In [0]:
gold_region_performance.show(5)

+------+--------+--------------------+------------+-------------+-------------------+---------------+-----------------+-----------+
|region|store_id|          store_name|total_orders|total_revenue|total_refund_amount|returned_orders|  return_rate_pct|net_revenue|
+------+--------+--------------------+------------+-------------+-------------------+---------------+-----------------+-----------+
| SOUTH|    S206|     Grover and Sons|         199|    7734706.0|           698053.0|             14|7.035175879396985|  7036653.0|
|  EAST|    S208|            Saha LLC|         195|    7424864.0|           776998.0|             17|8.717948717948717|  6647866.0|
| SOUTH|    S201|            Kibe PLC|         167|    6998214.0|          1062054.0|             25|14.97005988023952|  5936160.0|
| SOUTH|    S228|Kashyap, Sachdev ...|         183|    7411695.0|           411076.0|             10| 5.46448087431694|  7000619.0|
|  WEST|    S200|Bhasin, Srivastav...|         194|    7085858.0|           

In [0]:
gold_region_performance.count()

43

Gold Table-4 (Customer Analytics Table)

In [0]:
sales_customers = sales_silver.alias("s").join(
    customers_silver.alias("c"),
    col("s.customer_id") == col("c.customer_id"),
    "inner"
)

In [0]:
sales_customers_sel = sales_customers.select(
    col("c.customer_id"),
    col("c.customer_name"),
    col("c.customer_segment"),
    col("s.order_id"),
    col("s.revenue")
)

In [0]:
gold_customer_analytics = sales_customers_sel.groupBy(
    "customer_id",
    "customer_name",
    "customer_segment"
).agg(
    countDistinct("order_id").alias("total_orders"),
    sum_("revenue").alias("total_spend"),
    avg("revenue").alias("avg_order_value")
)

In [0]:

gold_customer_analytics = gold_customer_analytics.withColumn(
    "customer_value_score",
    col("total_orders") * col("avg_order_value")
)

In [0]:
gold_customer_analytics.write.format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_BASE_PATH}/gold_customer_analytics")
print("gold_customer_analytics table written")

gold_customer_analytics table written


In [0]:
gold_customer_analytics.show(5)

+-----------+-----------------+----------------+------------+-----------+---------------+--------------------+
|customer_id|    customer_name|customer_segment|total_orders|total_spend|avg_order_value|customer_value_score|
+-----------+-----------------+----------------+------------+-----------+---------------+--------------------+
|      C5315|     Ritvik Dubey|       WHOLESALE|          12|   505827.0|       42152.25|            505827.0|
|      C6202|        Keya Sood|       WHOLESALE|           4|   198283.0|       49570.75|            198283.0|
|      C5212|Aaryahi Jayaraman|       WHOLESALE|           5|   132413.0|        26482.6|            132413.0|
|      C5063|     Saanvi Borde|          RETAIL|           5|   138037.0|        27607.4|            138037.0|
|      C6545|  Mamooty Sampath|       CORPORATE|           2|    34283.0|        17141.5|             34283.0|
+-----------+-----------------+----------------+------------+-----------+---------------+--------------------+
o

In [0]:
gold_customer_analytics.count()

1873

Gold Table-5 (Target_vs_Actual Table)

In [0]:
actual_sales = sales_silver.withColumn(
    "order_year", year("order_date")
).withColumn(
    "order_month", month("order_date")
).groupBy(
    "region", "order_year", "order_month"
).agg(
    sum_("revenue").alias("actual_revenue")
)

In [0]:
target_vs_actual = actual_sales.alias("a").join(
    targets_silver.alias("t"),
    (col("a.region") == col("t.region")) &
    (col("a.order_year") == col("t.year")) &
    (col("a.order_month") == col("t.month")),
    "left"
)

In [0]:
gold_target_vs_actual = target_vs_actual \
    .withColumn(
        "revenue_variance",
        col("actual_revenue") - col("sales_target")
    ) \
    .withColumn(
        "achievement_pct",
        when(col("sales_target") > 0,
             (col("actual_revenue") / col("sales_target")) * 100
        ).otherwise(0)
    )

In [0]:
gold_target_vs_actual = gold_target_vs_actual.select(
    col("a.region").alias("region"),
    col("a.order_year").alias("year"),
    col("a.order_month").alias("month"),
    col("actual_revenue"),
    col("sales_target"),
    col("revenue_variance"),
    col("achievement_pct")
)

In [0]:
gold_target_vs_actual.write.format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_BASE_PATH}/gold_target_vs_actual")
print("gold_target_vs_actual table written")

gold_target_vs_actual table written


In [0]:
gold_target_vs_actual.show(5)

+------+----+-----+--------------+------------+----------------+------------------+
|region|year|month|actual_revenue|sales_target|revenue_variance|   achievement_pct|
+------+----+-----+--------------+------------+----------------+------------------+
| SOUTH|2024|    4|     2639502.0|      942645|       1696857.0| 280.0101841096065|
| SOUTH|2024|    1|     3124497.0|      598884|       2525613.0| 521.7198990121626|
|  WEST|2024|    4|     3729236.0|      965847|       2763389.0|  386.110429498668|
|  WEST|2024|   12|     2663386.0|      541125|       2122261.0|492.19422499422495|
| NORTH|2025|    2|     5211564.0|        NULL|            NULL|               0.0|
+------+----+-----+--------------+------------+----------------+------------------+
only showing top 5 rows


In [0]:
gold_target_vs_actual.count()

100

Gold Delta Tables

In [0]:
(
    gold_sales_performance.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("misha_azure.default.gold_sales_performance")
)

In [0]:
(
    gold_product_performance.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("misha_azure.default.gold_product_performance")
)

In [0]:
(
    gold_region_performance.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("misha_azure.default.gold_region_performance")
)

In [0]:
(
    gold_customer_analytics.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("misha_azure.default.gold_customer_analytics")
)

In [0]:
(
    gold_target_vs_actual.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("misha_azure.default.gold_target_vs_actual")
)